# Setup

Git clone

In [ ]:
!git clone https://github.com/nicolo-monzu/federated-learning.git
%cd /content/federated-learning

Mount and link Google Drive (optional)

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

for model_folder in ["centralized_model", "federated_model", "federated_sparse_model"]:
    drive_folder = f"/content/drive/MyDrive/{model_folder}/checkpoints"
    repo_folder = f"/content/federated-learning/{model_folder}/checkpoints"

    os.makedirs(drive_folder, exist_ok=True)
    os.makedirs(repo_folder, exist_ok=True)
    shutil.rmtree(repo_folder)

    os.symlink(drive_folder, repo_folder)


def copy_results_on_drive(model_folder):
    # Copy logs and plots into Google Drive after the run has finished
    drive_root = "/content/drive/MyDrive"

    folders_to_copy = ["logs", "plots"]

    if os.path.isdir(drive_root):
        for folder in folders_to_copy:
            source_folder = f"/content/federated-learning/{model_folder}/{folder}"
            drive_folder = f"/content/drive/MyDrive/{model_folder}/{folder}"

            if os.path.isdir(source_folder):
                os.makedirs(drive_folder, exist_ok=True)
                shutil.copytree(source_folder, drive_folder, dirs_exist_ok=True)
                print(f"{folder} copied to: {drive_folder}")
            else:
                print(f"{folder} folder not found: {source_folder}")
    else:
        print("Google Drive is not mounted. Plots and logs were not copied.")

Download dataset from drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

!mkdir -p /content/federated-learning/dataset/
!cp /content/drive/MyDrive/cifar-100-python.tar.gz /content/federated-learning/dataset/
!tar -xzvf dataset/cifar-100-python.tar.gz

# Experiments

See transformed data

In [ ]:
%run utils/dataviewer.py

Train

In [ ]:
# !python train.py start -h
!python train.py start

Train Federated

In [ ]:
!python train_federated.py start [-n <run_name>] [-r <rounds>]

Train Federated Sparse

In [ ]:
!python train_federated_sparse.py start [-n <run_name>] [-r <rounds>]

Evaluate model

In [ ]:
!python eval.py <checkpoint>

Hyperparameter search

In [ ]:
!python hp_search.py

Federated learning experiments

In [ ]:
from train_federated import start

J = 4
Nc = 100

start(run_name=f"federated_j_{J}_nc_{Nc}",
      num_rounds = 192*4 // J,
      num_steps_per_client = J,
      num_classes_per_client = Nc,
      rounds_per_scheduler_step = 16 // J,
      scale_grow_interval = 48*4 // J,
      validation_interval = 64 // J
      )

# Copy logs and plots into Google Drive after the run has finished
try:
    copy_results_on_drive("federated_model")
except NameError:
    print("Google Drive is not mounted. Plots and logs were not copied.")

Sparsity sweep

In [ ]:
from masking import MaskRule
from train_federated_sparse import start

J = 16
Nc = 50
num_calibration_round = 10
mask_rule = MaskRule.LEAST_SENSITIVE

for sparsity in [0.2, 0.4, 0.6, 0.8]:
    start(run_name=f"fed_sparse_s_{sparsity}_cr_{num_calibration_round}_j_{J}_nc_{Nc}",
          num_rounds = 192*4 // J,
          num_steps_per_client = J,
          num_classes_per_client = Nc,
          rounds_per_scheduler_step = 16 // J,
          scale_grow_interval = 48*4 // J,
          validation_interval = 64 // J,
          sparsity = sparsity,
          num_calibration_round = num_calibration_round,
          mask_rule = mask_rule
    )

    # Copy logs and plots into Google Drive after the run has finished
    try:
        copy_results_on_drive("federated_sparse_model")
    except NameError:
        print("Google Drive is not mounted. Plots and logs were not copied.")

Number of round ablation

In [ ]:
from masking import MaskRule
from train_federated_sparse import start

J = 16
Nc = 50
sparsity = 0.4
mask_rule = MaskRule.LEAST_SENSITIVE

for num_calibration_round in [5, 3, 1]:
    start(run_name=f"fed_sparse_s_{sparsity}_cr_{num_calibration_round}_j_{J}_nc_{Nc}",
          num_rounds = 192*4 // J,
          num_steps_per_client = J,
          num_classes_per_client = Nc,
          rounds_per_scheduler_step = 16 // J,
          scale_grow_interval = 48*4 // J,
          validation_interval = 64 // J,
          sparsity = sparsity,
          num_calibration_round = num_calibration_round,
          mask_rule = mask_rule
    )

    # Copy logs and plots into Google Drive after the run has finished
    try:
        copy_results_on_drive("federated_sparse_model")
    except NameError:
        print("Google Drive is not mounted. Plots and logs were not copied.")

Try different mask calibration rules

In [ ]:
from masking import MaskRule
from train_federated_sparse import start

J = 16
Nc = 50
sparsity = 0.4
num_calibration_round = 5

for mask_rule in range(1, 5, 1):
    mask_rule_name = MaskRule(mask_rule).name.lower()
    start(run_name=f"fed_sparse_{mask_rule_name}_j_{J}_nc_{Nc}",
          num_rounds = 192*4 // J,
          num_steps_per_client = J,
          num_classes_per_client = Nc,
          rounds_per_scheduler_step = 16 // J,
          scale_grow_interval = 48*4 // J,
          validation_interval = 64 // J,
          sparsity = sparsity,
          num_calibration_round = num_calibration_round,
          mask_rule = mask_rule
    )

    # Copy logs and plots into Google Drive after the run has finished
    try:
        copy_results_on_drive("federated_sparse_model")
    except NameError:
        print("Google Drive is not mounted. Plots and logs were not copied.")

Create a plot from a log file

In [ ]:
from plot import plot_training
plot_training(
    run_name="",
    logs_dir="centralized_model/logs/",
    save_dir="centralized_model/plots/", # If None, the plots will be printed
    federated=False
)